In [179]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [180]:
path_db = '../db/filmes.db'

engine = create_engine(f'sqlite:///{path_db}')

In [181]:
df = pd.read_sql('SELECT * from filmes', con=engine)
df

,id,title,original_title,original_language,overview,budget,revenue,runtime,release_date,genres,popularity,vote_average,vote_count,num_genres,release_year
0,411405,Small Crimes,Small Crimes,en,"A disgraced former cop, fresh off a six-year p...",8.201666e+06,4.243626e+06,95.0,2017-04-28 00:00:00.000000,"['Drama', 'Comedy', 'Thriller', 'Crime']",7.219022,5.8,55.0,4,2017.0
1,42492,Up the Sandbox,Up the Sandbox,en,"A young wife and mother, bored with day-to-day...",3.543621e+05,2.978317e+06,97.0,1972-12-21 00:00:00.000000,"['Drama', 'Comedy']",0.138450,7.3,2.0,2,1972.0
2,12143,Bad Lieutenant,Bad Lieutenant,en,"While investigating a young nun's rape, a corr...",1.000000e+06,2.019469e+06,96.0,1992-09-16 00:00:00.000000,"['Crime', 'Drama']",6.417037,6.9,162.0,2,1992.0
3,9976,Satan's Little Helper,Satan's Little Helper,en,A naïve young boy unknowingly becomes the pawn...,1.890390e+06,1.710526e+06,100.0,2004-01-01 00:00:00.000000,"['Horror', 'Romance', 'Comedy']",2.233189,5.0,42.0,3,2004.0
4,46761,Sitcom,Sitcom,fr,The adventures of an upper-class suburban fami...,8.550376e+05,2.725346e+06,80.0,1998-05-27 00:00:00.000000,"['Comedy', 'Drama', 'Thriller']",1.800582,6.4,27.0,3,1998.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4993,3093,Basic Instinct 2,Basic Instinct 2,en,Novelist Catherine Tramell is once again in tr...,7.000000e+07,3.862948e+07,114.0,2006-03-29 00:00:00.000000,"['Crime', 'Mystery', 'Thriller']",15.321902,4.6,183.0,3,2006.0
4994,246127,Every Thing Will Be Fine,Every Thing Will Be Fine,en,"One day, driving aimlessly around the outskirt...",9.853274e+06,8.034000e+03,118.0,2015-04-02 00:00:00.000000,['Drama'],5.723103,5.3,79.0,1,2015.0
4995,9803,Seven Dwarfs,7 Zwerge - Männer allein im Wald,de,The Seven Dwarves live deep within a female-fr...,9.674091e+06,1.694475e+06,95.0,2004-10-28 00:00:00.000000,['Comedy'],4.582736,5.2,70.0,1,2004.0
4996,336970,True Siblings,Syskonsalt,sv,"The siblings Linus, 19-years-old, who are taki...",1.060106e+06,4.436564e+06,58.0,2000-09-13 00:00:00.000000,"['Drama', 'TV Movie']",2.364355,8.0,2.0,2,2000.0


In [182]:
df['revenue'].median()

2000145.379574506

In [183]:
df.columns

Index(['id', 'title', 'original_title', 'original_language', 'overview',
       'budget', 'revenue', 'runtime', 'release_date', 'genres', 'popularity',
       'vote_average', 'vote_count', 'num_genres', 'release_year'],
      dtype='object')

In [184]:
features = ['budget', 'runtime', 'popularity', 'vote_average', 'vote_count', 'num_genres', 'release_year']

In [185]:
def modelo_linear(df, features, coluna_alvo):
    
    valido = ~df[features].isna().any(axis=1) & (df[coluna_alvo] > 0)
    
    X = df.loc[valido, features]
    y = df.loc[valido, coluna_alvo]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state=42)
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
    
    param_grid = {
        'model__fit_intercept': [True, False],
        'model__positive': [False, True]
    }
    
    scoring = {
        'r2': 'r2',
        'rmse': 'neg_root_mean_squared_error'
    }
    
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=scoring,
        cv=kf,
        n_jobs=-1,
        verbose=2,
        refit='rmse'
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f'Melhores parâmetros: {grid_search.best_params_}')
    print(f'RMSE (médio): {grid_search.best_score_}')
    
    best_model = grid_search.best_estimator_
    best_model.fit(X_train, y_train)
    
    y_pred = best_model.predict(X_test)
    
    print(f'RMSE no treino final do modelo: {root_mean_squared_error(y_test, y_pred)}')
    print(f'R2 no treino final do modelo: {r2_score(y_test, y_pred)}')
    
    return {'gs': grid_search,
            'bm': best_model}

In [186]:
linear_model = modelo_linear(df, features, 'revenue')
linear_model

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Melhores parâmetros: {'model__fit_intercept': True, 'model__positive': True}
RMSE (médio): -385698332.68846387
RMSE no treino final do modelo: 204961667.7703516
R2 no treino final do modelo: 0.11852995015167445


{'gs': GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
              estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                        ('model', LinearRegression())]),
              n_jobs=-1,
              param_grid={'model__fit_intercept': [True, False],
                          'model__positive': [False, True]},
              refit='rmse',
              scoring={'r2': 'r2', 'rmse': 'neg_root_mean_squared_error'},
              verbose=2),
 'bm': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model', LinearRegression(positive=True))])}

Temos, portanto, que o modelo apresentou um viés muito alto no k-fold dado pelo gride search com RMSE médio bastante elevado de aproximadamente __385 milhões__ tendo em vista que o valor típico em `revenue` é de __2 milhões__, o que mostra que o modelo tem baixa capacidade preditiva. Porém, no treino final dado pelo melhor modelo do grid search tivemos uma redução do RMSE para aproximadamente __205 milhões__ o que configura um viés bastante alto mas o modelo apresentou um grau de generalização dos dados, o que diz que aprendeu algo de útil. Pelo coeficiente de determinação `r2`, o modelo explica apenas aproximadamente __12%__ da variação da variavel `revenue` (variável dependente) dado pelas features (variáveis independentes), ou seja, a regressão linear não está capturando bem as relações nos dados. Com essas observações, vemos que esse modelo é muito simples para o dataset que pode ter relações não lineares entre os dados, outliers e features irrelevantes.

In [187]:
coeficientes = linear_model['bm'].named_steps['model'].coef_

df_coeficientes = pd.DataFrame({
    'Feature': features,
    'Coeficiente': coeficientes
})

print(df_coeficientes)

        Feature   Coeficiente
0        budget  4.943924e+05
1       runtime  7.584539e+06
2    popularity  0.000000e+00
3  vote_average  0.000000e+00
4    vote_count  1.844814e+08
5    num_genres  0.000000e+00
6  release_year  0.000000e+00


In [188]:
def modelo_linear_ridge(df, features, coluna_alvo):
    
    valido = ~df[features].isna().any(axis=1) & (df[coluna_alvo] > 0)
    
    X = df.loc[valido, features]
    y = df.loc[valido, coluna_alvo]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state=42)
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', Ridge())])
    
    param_grid = {
        'model__alpha': np.logspace(-3, 3, 13),
        'model__fit_intercept': [True, False],
        'model__solver': ['auto', 'cholesky', 'lsqr', 'sag']
    }
    
    scoring = {
        'r2': 'r2',
        'rmse': 'neg_root_mean_squared_error'
    }
    
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=scoring,
        cv=kf,
        n_jobs=-1,
        verbose=2,
        refit='rmse'
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f'Melhores parâmetros: {grid_search.best_params_}')
    print(f'RMSE (médio): {grid_search.best_score_}')
    
    best_model = grid_search.best_estimator_
    best_model.fit(X_train, y_train)
    
    y_pred = best_model.predict(X_test)
    
    print(f'RMSE no treino final do modelo: {root_mean_squared_error(y_test, y_pred)}')
    print(f'R2 no treino final do modelo: {r2_score(y_test, y_pred)}')
    
    return {'gs': grid_search,
            'bm': best_model}

In [189]:
ridge_model = modelo_linear_ridge(df, features, 'revenue')
ridge_model

Fitting 5 folds for each of 104 candidates, totalling 520 fits
Melhores parâmetros: {'model__alpha': 1000.0, 'model__fit_intercept': True, 'model__solver': 'auto'}
RMSE (médio): -354884709.0360614
RMSE no treino final do modelo: 190081965.92306682
R2 no treino final do modelo: 0.24186927436358874


{'gs': GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
              estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                        ('model', Ridge())]),
              n_jobs=-1,
              param_grid={'model__alpha': array([1.00000000e-03, 3.16227766e-03, 1.00000000e-02, 3.16227766e-02,
        1.00000000e-01, 3.16227766e-01, 1.00000000e+00, 3.16227766e+00,
        1.00000000e+01, 3.16227766e+01, 1.00000000e+02, 3.16227766e+02,
        1.00000000e+03]),
                          'model__fit_intercept': [True, False],
                          'model__solver': ['auto', 'cholesky', 'lsqr', 'sag']},
              refit='rmse',
              scoring={'r2': 'r2', 'rmse': 'neg_root_mean_squared_error'},
              verbose=2),
 'bm': Pipeline(steps=[('scaler', StandardScaler()), ('model', Ridge(alpha=1000.0))])}

Na Ridge temos ainda RMSE bastante elevado embora menores do que no modelo linear padrão. Ainda assim, o modelo generaliza no treino final para os dados de teste mostrando que está aprendendo algo útil e é o que se espera de um modelo regularizado. O `r2` um pouco maior diz que com a regularização Ridge o modelo explicou melhor a variação de `y` com relação as features. Isso nos diz que o modelo coseguiu capturar melhor as relações entre as features mas ainda é um modelo simples para este dataset.

In [191]:
coeficientes_l2 = ridge_model['bm'].named_steps['model'].coef_

df_coeficientes_l2 = pd.DataFrame({
    'Feature': features,
    'Coeficiente': coeficientes_l2
})

print(df_coeficientes_l2)

        Feature   Coeficiente
0        budget  2.697287e+07
1       runtime  8.931939e+06
2    popularity -4.170424e+05
3  vote_average -3.453387e+06
4    vote_count  1.352954e+08
5    num_genres -6.580850e+06
6  release_year -6.913305e+06


In [192]:
def modelo_linear_lasso(df, features, coluna_alvo):
    
    valido = ~df[features].isna().any(axis=1) & (df[coluna_alvo] > 0)
    
    X = df.loc[valido, features]
    y = df.loc[valido, coluna_alvo]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state=42)
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', Lasso())])
    
    param_grid = {
        'model__alpha': np.logspace(-4, 1, 10),
        'model__fit_intercept': [True, False],
        'model__max_iter': [1000, 5000, 10000]
    }
    
    scoring = {
        'r2': 'r2',
        'rmse': 'neg_root_mean_squared_error'
    }
    
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=scoring,
        cv=kf,
        n_jobs=-1,
        verbose=2,
        refit='rmse'
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f'Melhores parâmetros: {grid_search.best_params_}')
    print(f'RMSE (médio): {grid_search.best_score_}')
    
    best_model = grid_search.best_estimator_
    best_model.fit(X_train, y_train)
    
    y_pred = best_model.predict(X_test)
    
    print(f'RMSE no treino final do modelo: {root_mean_squared_error(y_test, y_pred)}')
    print(f'R2 no treino final do modelo: {r2_score(y_test, y_pred)}')
    
    return {'gs': grid_search,
            'bm': best_model}

In [193]:
lasso_model = modelo_linear_lasso(df, features, 'revenue')
lasso_model

Fitting 5 folds for each of 60 candidates, totalling 300 fits
Melhores parâmetros: {'model__alpha': 10.0, 'model__fit_intercept': True, 'model__max_iter': 1000}
RMSE (médio): -391766115.6029151
RMSE no treino final do modelo: 205898733.99634588
R2 no treino final do modelo: 0.11045152250169621


{'gs': GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
              estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                        ('model', Lasso())]),
              n_jobs=-1,
              param_grid={'model__alpha': array([1.00000000e-04, 3.59381366e-04, 1.29154967e-03, 4.64158883e-03,
        1.66810054e-02, 5.99484250e-02, 2.15443469e-01, 7.74263683e-01,
        2.78255940e+00, 1.00000000e+01]),
                          'model__fit_intercept': [True, False],
                          'model__max_iter': [1000, 5000, 10000]},
              refit='rmse',
              scoring={'r2': 'r2', 'rmse': 'neg_root_mean_squared_error'},
              verbose=2),
 'bm': Pipeline(steps=[('scaler', StandardScaler()), ('model', Lasso(alpha=10.0))])}

A Lasso foi o pior dos três modelos sendo pior até mesmo do que o modelo linear padrão devido a RMSE médio mais alto (aumentou o viés), `r2` menor e capacidade de gerneralização mais baixa.

In [194]:
coeficientes_l1 = lasso_model['bm'].named_steps['model'].coef_

df_coeficientes_l1 = pd.DataFrame({
    'Feature': features,
    'Coeficiente': coeficientes_l1
})

print(df_coeficientes_l1)

        Feature   Coeficiente
0        budget  3.532872e+06
1       runtime  1.043029e+07
2    popularity -2.074382e+07
3  vote_average -7.766877e+06
4    vote_count  1.947809e+08
5    num_genres -7.389983e+06
6  release_year -9.453362e+06


Com essas observações vemos que o modelo com melhor desempenho foi o da regressão Ridge. 

Com relação aos coeficientes do melhor modelo (treino final) dado pelo grid search, não tivemos coeficientes zerados em nenhum dos modelos regularizados (Ridge e Lasso) ao passo que no modelo linear, temos algumas features iguais a zero.

O __overfitting__ é quando o modelo se ajusta demais aos dados de treinamento, desempenhando bem nesses dados porém gerando baixa capacidade de generalização do modelo para novas instâncias. Nessa situação, dizemos que o modelo apenas "decorou" os dados de treinamento (alta variância). A regularização tem como objetivo diminuir a complexidade do modelo que por consequência irá dimuir a variância e, portanto o overfitting, aceitando um pouco do aumento do viés. Por meio disso, o modelo tem uma queda no desempeho do treino mas aumenta a capacidade de generalização para novos dados, que é o ideal. O processo de regularização se da através da inclusão de um novo termo na função de custo a ser minimizada no treinamento. Esse termo faz com que os parâmetros treinaveis reduzam e, em alguns casos, como na __Regressão Lasso__, são zerados, fazendo uma seleção de features. A redução dos pesos deixa o modelo mais simples e menos sensíveis a pequenas flutuações nos dados de treinamento.    